### **SILVER Layer Notebook : Data Cleaning & Integration**
### 

**Load Bronze Data**

In [1]:
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date, col


StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 3, Finished, Available, Finished)

In [2]:
# df_orders is a Spark DataFrame containing CSV data from "Files/ShoppingMart_Bronze_Orders/ShoppingMart_orders.csv".

df_orders = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Orders/ShoppingMart_orders.csv")

display(df_orders)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 061e1e6e-6b06-4748-b705-ce1d942910fe)

In [3]:
## Check the schema
df_orders.printSchema()

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 5, Finished, Available, Finished)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- TotalAmount: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)



In [4]:
## Drop Null Values from the columns

df_orders = df_orders.dropna(subset= ["OrderID", "OrderDate","CustomerID","Quantity","TotalAmount"])

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 6, Finished, Available, Finished)

In [5]:

# Verify schema
# df_orders.printSchema()

#Convert OrderDate & TotalAmount Data types

df_orders = df_orders.withColumn("OrderDate", to_date(col("OrderDate")))

df_orders = df_orders.withColumn("TotalAmount", col("TotalAmount").cast("float"))

df_orders = df_orders.withColumn("Quantity", col("Quantity").cast("int"))

df_orders.printSchema()


StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 7, Finished, Available, Finished)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- TotalAmount: float (nullable = true)
 |-- PaymentMethod: string (nullable = true)



In [6]:
display(df_orders)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e5edf40e-daf4-4712-a76f-31fdab305af2)

### **Reading other tables**

In [7]:
df_customers = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Customers/ShoppingMart_customers.csv")
# df now is a Spark DataFrame containing CSV data from "Files/ShoppingMart_Bronze_Customers/ShoppingMart_customers.csv".
display(df_customers)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5d33dc57-303f-4191-bf0e-d00321a09d0d)

In [8]:
df_products = spark.read.format("csv").option("header","true").load("Files/ShoppingMart_Bronze_Products/ShoppingMart_products.csv")
# df now is a Spark DataFrame containing CSV data from "Files/ShoppingMart_Bronze_Products/ShoppingMart_products.csv".
display(df_products)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9eef828c-366f-4c57-9c5d-f36aa8db4caf)

In [9]:
#Joining Tables

df_orders = df_orders \
    .join(df_customers, on = 'CustomerID',how =  "inner") \
    .join(df_products, on = 'ProductID',how =  "inner")


display(df_orders)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 64105ea8-c366-482f-9e8f-703dfac037cc)

In [10]:
df_orders.count()

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 12, Finished, Available, Finished)

2939

In [11]:
df_orders.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Orders/ShoppingMart_OrderData")

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 13, Finished, Available, Finished)

### **Reading Unstructured Data**


In [12]:
df_reviews = spark.read.json("Files/ShoppingMart_Bronze_Reviews/ShoppingMart_review.json")
display(df_reviews)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ff0db1ab-5eb6-4b34-a851-c784f2646654)

In [13]:
df_socialmedia = spark.read.json("Files/ShoppingMart_Bronze_Social_Media/ShoppingMart_social_media.json")

display(df_socialmedia)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, bd4fc168-4ef7-4ad7-a336-b5f795610e99)

In [14]:
df_web_logs = spark.read.json("Files/ShoppingMart_Bronze_Web_Logs/ShoppingMart_web_logs.json")
# df now is a Spark DataFrame containing JSON data from "abfss://ShoppingMart_Analytics@onelake.dfs.fabric.microsoft.com/SM_Bro_LH.Lakehouse/Files/ShoppingMart_Bronze_Web_Logs/ShoppingMart_web_logs.json".
display(df_web_logs)

StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 16, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 6521dc74-c65e-43ff-95ce-984b280751d9)

### **Write Unstructured Data to Silver Layer**

In [15]:
df_reviews.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_Reviews/ShoppingMart_ReviewsData")
df_socialmedia.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_SocialMedia/ShoppingMart_SocialMediaData")
df_web_logs.write.mode("overwrite").parquet("Files/ShoppingMart_Silver_web_logs/ShoppingMart_web_logsData")



StatementMeta(, bb2e8c45-155d-4e87-989c-5e59f9ecc4b4, 17, Finished, Available, Finished)